# NB_08 — Scenario Configuration

In [1]:
# ============================================================
# NB_08 — SCENARIO CONFIGURATION
# 08.01 — IMPORTS & CONFIGURATION
# ============================================================

import pandas as pd
import numpy as np

from pathlib import Path

# Directorios
PROJECT_DIR = Path(r"C:\ds4b\C2011-13 Wind Farm_v4")
OUTPUT_DIR = PROJECT_DIR / "07_Outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Parámetros generales
FLOAT_TOLERANCE = 1e-7

print("NB_08 — SCENARIO CONFIGURATION")
print(f"Project directory: {PROJECT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

In [2]:
# ============================================================
# 08.02 — LOAD FROZEN BASELINE
# ============================================================

# 1. CPM: calendario y holguras
df_cpm_base = pd.read_csv(
    OUTPUT_DIR / "cpm_activity_schedule.csv"
)

df_cpm_base = df_cpm_base.rename(columns={
    "ES_Days": "ES",
    "EF_Days": "EF",
    "LS_Days": "LS",
    "LF_Days": "LF",
    "Total_Float_Days": "Total_Float",
    "Free_Float_Days": "Free_Float"
})

# 2. Registro de seguimiento
df_tracking_base = pd.read_excel(
    OUTPUT_DIR / "Activity_Tracking_Register.xlsx",
    sheet_name="Activity Tracking"
)

# 3. Control de calendario y suministros
df_schedule_base = pd.read_excel(
    OUTPUT_DIR / "Schedule_Control.xlsx",
    sheet_name="Schedule Control"
)

# 4. Resumen de línea base
df_project_summary = pd.read_csv(
    OUTPUT_DIR / "cpm_project_summary.csv"
)

BASELINE_DURATION = float(
    df_project_summary.loc[0, "Project_Duration_CPM_Days"]
)

BASELINE_CRITICAL = int(
    df_project_summary.loc[0, "Critical_Activities"]
)

# 5. Validaciones
assert len(df_cpm_base) == 135
assert df_cpm_base["ID"].is_unique
assert df_schedule_base["ID"].is_unique
assert df_cpm_base["Is_Critical"].sum() == BASELINE_CRITICAL

print("BASELINE LOADED SUCCESSFULLY")
print(f"CPM activities: {len(df_cpm_base)}")
print(f"Tracking activities: {len(df_tracking_base)}")
print(f"Schedule control activities: {len(df_schedule_base)}")
print(f"Baseline duration: {BASELINE_DURATION:.0f} days")
print(f"Critical activities: {BASELINE_CRITICAL}")

In [3]:
# ============================================================
# 08.03 — DIAGNOSTICAR ACTIVITY CATALOG
# ============================================================

print("CPM BASE")
print(df_cpm_base[["ID", "Name", "Duration_Plan_Days"]].head(10))

print("\nSCHEDULE BASE")
print(df_schedule_base[["ID", "Material_Type", "Material_Status",
                        "Supply_Delay_Days", "Buffer_Days",
                        "Material_Alarm"]].head(10))

print("\nTRACKING BASE")
print(df_tracking_base[["ID", "Name"]].head(10))

print("\nDUPLICATED COLUMNS")
print("CPM:", df_cpm_base.columns[df_cpm_base.columns.duplicated()].tolist())
print("Schedule:", df_schedule_base.columns[df_schedule_base.columns.duplicated()].tolist())

In [4]:
# ============================================================
# 08.03 — BUILD ACTIVITY CATALOG
# ============================================================

cpm_columns = [
    "ID",
    "Name",
    "Duration_Plan_Days",
    "ES",
    "EF",
    "Total_Float",
    "Is_Critical"
]

material_columns = [
    "ID",
    "Material_Type",
    "Material_Status",
    "Supply_Delay_Days",
    "Buffer_Days",
    "Material_Alarm"
]

activity_catalog = (
    df_cpm_base[cpm_columns]
    .merge(
        df_schedule_base[material_columns],
        on="ID",
        how="left",
        validate="one_to_one"
    )
    .sort_values("ID")
    .reset_index(drop=True)
)

# Validaciones de integridad
assert len(activity_catalog) == 135
assert activity_catalog["ID"].is_unique
assert activity_catalog["Name"].notna().all()
assert activity_catalog["Duration_Plan_Days"].notna().all()

print("ACTIVITY CATALOG VALIDATED")
print(f"Activities: {len(activity_catalog)}")
print(f"Unique IDs: {activity_catalog['ID'].nunique()}")
print(f"Missing material records: {activity_catalog['Material_Type'].isna().sum()}")

display(activity_catalog.head(10))

In [5]:
# ============================================================
# 08.03.1 — MATERIAL DATA QUALITY CHECK
# ============================================================

print("MISSING VALUES")
display(
    activity_catalog[
        ["Material_Type", "Material_Status",
         "Supply_Delay_Days", "Buffer_Days", "Material_Alarm"]
    ].isna().sum()
)

print("\nMATERIAL TYPE")
display(activity_catalog["Material_Type"].value_counts(dropna=False))

print("\nMATERIAL STATUS")
display(activity_catalog["Material_Status"].value_counts(dropna=False))

print("\nBUFFER DISTRIBUTION")
display(activity_catalog["Buffer_Days"].value_counts(dropna=False).sort_index())

print("\nACTIVITIES WITH MATERIAL RISK")
display(
    activity_catalog[
        activity_catalog["Material_Status"] != "Non-Critical"
    ][
        ["ID", "Name", "Material_Type", "Material_Status",
         "Supply_Delay_Days", "Buffer_Days", "Material_Alarm"]
    ].head(15)
)

In [6]:
# ============================================================
# 08.04 — SCENARIO DEFINITION
# ============================================================

scenario_records = [
    {
        "Scenario_ID": "SC01",
        "Scenario_Name": "Supplier Delay",
        "Activity_ID": 79,
        "Impact_Type": "SUPPLY_DELAY",
        "Delay_Days": 15,
        "Description": "Hypothetical delay in critical material delivery."
    },
    {
        "Scenario_ID": "SC02",
        "Scenario_Name": "Activity Duration Increase",
        "Activity_ID": 20,
        "Impact_Type": "ACTIVITY_DELAY",
        "Delay_Days": 7,
        "Description": "Hypothetical increase in activity execution duration."
    },
    {
        "Scenario_ID": "SC03",
        "Scenario_Name": "Combined Project Disruption",
        "Activity_ID": 79,
        "Impact_Type": "SUPPLY_DELAY",
        "Delay_Days": 15,
        "Description": "Combined scenario: supplier disruption."
    },
    {
        "Scenario_ID": "SC03",
        "Scenario_Name": "Combined Project Disruption",
        "Activity_ID": 20,
        "Impact_Type": "ACTIVITY_DELAY",
        "Delay_Days": 7,
        "Description": "Combined scenario: activity duration increase."
    }
]

df_scenarios = pd.DataFrame(scenario_records)

display(df_scenarios)

In [7]:
# ============================================================
# 08.05 — SCENARIO VALIDATION
# ============================================================

valid_impact_types = {
    "SUPPLY_DELAY",
    "ACTIVITY_DELAY"
}

assert df_scenarios["Scenario_ID"].notna().all()
assert df_scenarios["Activity_ID"].isin(
    activity_catalog["ID"]
).all()

assert df_scenarios["Impact_Type"].isin(
    valid_impact_types
).all()

assert (df_scenarios["Delay_Days"] > 0).all()

assert df_scenarios[
    ["Scenario_ID", "Activity_ID", "Impact_Type"]
].duplicated().sum() == 0

print("SCENARIO VALIDATION: PASSED")

print("\nScenarios:")
display(
    df_scenarios.groupby("Scenario_ID").agg(
        Affected_Activities=("Activity_ID", "nunique"),
        Total_Impact_Days=("Delay_Days", "sum")
    )
)

In [8]:
"""
# ============================================================
# 08.06 — EXPORT SCENARIOS
# ============================================================

scenario_file = OUTPUT_DIR / "Scenario_Configuration.xlsx"

with pd.ExcelWriter(scenario_file, engine="openpyxl") as writer:
    activity_catalog.to_excel(
        writer,
        sheet_name="Activity Catalog",
        index=False
    )

    df_scenarios.to_excel(
        writer,
        sheet_name="Scenarios",
        index=False
    )

print("SCENARIO CONFIGURATION EXPORTED")
print(scenario_file)
"""

In [9]:
# ============================================================
# 08.06 — FINAL SCENARIO EXPORT
# ============================================================

scenario_file = OUTPUT_DIR / "Scenario_Configuration.xlsx"

# Comprobar que existe la tabla de escenarios
assert "df_scenarios" in globals(), (
    "df_scenarios no existe. Ejecuta primero la celda de configuración."
)

assert len(df_scenarios) == 4
assert len(activity_catalog) == 135

with pd.ExcelWriter(
    scenario_file,
    engine="openpyxl"
) as writer:

    activity_catalog.to_excel(
        writer,
        sheet_name="Activity Catalog",
        index=False
    )

    df_scenarios.to_excel(
        writer,
        sheet_name="Scenarios",
        index=False
    )

# Verificar exportación
xls_check = pd.ExcelFile(scenario_file)

assert "Activity Catalog" in xls_check.sheet_names
assert "Scenarios" in xls_check.sheet_names

print("NB_08 EXPORT VERIFIED")
print("Sheets:", xls_check.sheet_names)
print("Activities:", len(pd.read_excel(
    scenario_file, sheet_name="Activity Catalog"
)))
print("Scenario records:", len(pd.read_excel(
    scenario_file, sheet_name="Scenarios"
)))